# M2 — conditional low-rank Gaussian copula

This self-contained notebook explains how frozen per-entity forecast information predicts an origin-specific correlation matrix. See [Chapter 4](../../docs/scientific/04_dependence_models.md) and [Chapter 5](../../docs/scientific/05_training_sampling_scoring.md).

## Notation and scientific sample

$i$ indexes origin $t^{(i)}$, $g$ indexes $\mathcal E_g=\{k_1,\ldots,k_{K_g}\}$, and $\tau$ is one-based lead. $Y_{k,\tau}^{(i)}$ is random, $y_{k,\tau}^{(i)}$ is observed, $F_{k,\tau}^{(i)}$ is fixed, and $A_{g,\tau}^{(i)}=\sum_kY_{k,\tau}^{(i)}$. Define $V_{k,\tau}^{(i)}\in\{0,1\}$ and $V_{g,\tau}^{(i)}=\prod_{k\in\mathcal E_g}V_{k,\tau}^{(i)}$. When this product is one, $X_{g,\tau}^{(i)}=[v_{k,\tau}^{(i)}]_k\in\mathbb R^{K_g\times F}$ contains origin-measurable features and $\mathbf z_{g,\tau}^{(i)}\in\mathbb R^{K_g}$ contains Gaussianized pseudo-PIT scores. $X$ is a feature matrix; $V$ is validity.

The fixed marginal law concerns $Y_{k,\tau}^{(i)}$; M2 predicts only $R_{g,\tau}^{(i)}$, the copula correlation joining those marginals.

## Low-rank covariance construction

A shared entity-wise network returns $(\lambda_k,b_k)=f_\theta(v_k)$, with $\lambda_k\in\mathbb R^r$ and $\sigma_k=\operatorname{softplus}(b_k)+\sigma_{\min}>0$. Stacking rows gives $\Lambda\in\mathbb R^{K_g\times r}$ and
$$\Sigma=\Lambda\Lambda^\mathsf T+\operatorname{diag}(\sigma_1^2,\ldots,\sigma_{K_g}^2),\qquad R_{ab}=\Sigma_{ab}/\sqrt{\Sigma_{aa}\Sigma_{bb}}.$$
Aligned loading vectors imply positive shared covariance; opposing vectors imply negative covariance. The uniqueness term keeps full rank. Individual factor coordinates are rotation non-identifiable, so scientific interpretation belongs to $R$, not one column of $\Lambda$.

In [ ]:
from pathlib import Path
import sys, numpy as np, matplotlib.pyplot as plt
PROJECT_ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'pyproject.toml').is_file())
if str(PROJECT_ROOT/'src') not in sys.path: sys.path.insert(0,str(PROJECT_ROOT/'src'))
from simcast.config import load_base_config, load_method_config, resolve_run_config
from simcast.experiments import locate_compatible_cache
from simcast.fm.cache import load_pit_library
from simcast.cli.train_dependence import train_from_config
from simcast.cli.evaluate import evaluate_from_config
BASE_FILE='configs/bases/liander2024/transformer.yaml'
METHOD_FILE='configs/methods/m2_conditional_low_rank.yaml'
BASE_OVERRIDES=(); METHOD_OVERRIDES=()
TRAIN_IF_MISSING=False; RUN_EVALUATION=False
RUN_DIR=PROJECT_ROOT/'runs'/'notebook_walkthrough'/'m2_conditional_low_rank'
base=load_base_config(PROJECT_ROOT/BASE_FILE,overrides=BASE_OVERRIDES)
method=load_method_config(PROJECT_ROOT/METHOD_FILE,overrides=METHOD_OVERRIDES)
config=resolve_run_config(base,method)
cache_dir=locate_compatible_cache(base)
library=load_pit_library(cache_dir,access='training'); ds=library.dataset
entity_ids=[str(x) for x in ds.entity_id.values]; K_g=len(entity_ids)
print({'g':base.data.entity_type,'K_g':K_g,'rank r':method.model.latent_rank})


In [ ]:
Lambda=np.array([[.8,.1],[.7,.2],[-.4,.6]])
sigma=np.array([.3,.3,.3])
Sigma=Lambda@Lambda.T+np.diag(sigma**2); R=Sigma/np.sqrt(np.outer(np.diag(Sigma),np.diag(Sigma)))
fig,axes=plt.subplots(1,2,figsize=(9,3.6),constrained_layout=True)
axes[0].quiver(np.zeros(3),np.zeros(3),Lambda[:,0],Lambda[:,1],angles='xy',scale_units='xy',scale=1)
axes[0].set(xlim=(-.6,1),ylim=(-.1,.8),xlabel='factor 1',ylabel='factor 2',title='Illustrative loading vectors')
im=axes[1].imshow(R,vmin=-1,vmax=1,cmap='coolwarm'); axes[1].set(title='Implied normalized correlation',xlabel='entity',ylabel='entity'); fig.colorbar(im,ax=axes[1]); plt.show()


## Estimation

For one complete score vector, M2 minimizes $\mathcal L=\tfrac12[\log|R|+\mathbf z^\mathsf T(R^{-1}-I)\mathbf z]$. Training uses chronological training cases; checkpoint selection uses chronological validation pseudo-NLL. The word “pseudo” records that $z$ arose from finite quantile cells. M2's shared network processes each $v_k$ separately before the low-rank interaction, so it is permutation equivariant but not set-contextual before producing loadings.

In [ ]:
if TRAIN_IF_MISSING and not (RUN_DIR/'best.pt').is_file():
    train_from_config(config,cache_dir=cache_dir,output_dir=RUN_DIR)
if RUN_EVALUATION:
    evaluate_from_config(config,methods=('conditional_low_rank',),method_runs={'conditional_low_rank':RUN_DIR},cache_dir=cache_dir,output_dir=RUN_DIR/'evaluation')


## Controls and interpretation

`method.features.*` determines $v_k$. `method.model.latent_rank`, `hidden_dims`, `dropout`, `sigma_floor`, and `jitter` determine the map and covariance. `method.optimization.*` controls estimation. `ConditionalLowRankGaussianCopula` implements the matrix; `gaussian_copula_pseudo_nll` implements the objective. A gain over M1 supports predictable context variation under this parameterization; failure to gain is equally informative and does not imply absence of dependence.